# Tiền xử lý dữ liệu ADXL355

Notebook này thực hiện:
1. Đọc dữ liệu thô từ cảm biến ADXL355
2. Chuyển đổi Unix timestamp sang UTC datetime
3. Tính toán thời gian tương đối
4. Lọc nhiễu và loại bỏ DC offset
5. Lưu dữ liệu đã xử lý

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from datetime import datetime, timezone
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Đọc dữ liệu thô

In [ ]:
input_file = '../data/raw/sensor_B/axdl355_1.csv'
df = pd.read_csv(input_file)

print(f"Số mẫu: {len(df)}")
print(f"Các cột: {list(df.columns)}")
df.head()

## 2. Chuyển đổi thời gian sang UTC

In [ ]:
# Chuyển Unix timestamp sang UTC datetime
df['datetime_utc'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Tính thời gian tương đối (giây từ điểm bắt đầu)
df['time_s'] = df['timestamp'] - df['timestamp'].iloc[0]

# Thông tin thời gian
start_utc = df['datetime_utc'].iloc[0]
end_utc = df['datetime_utc'].iloc[-1]
duration = df['time_s'].iloc[-1]

# Tính tần số lấy mẫu
dt = np.diff(df['timestamp'])
fs = 1.0 / np.mean(dt)

print(f"=" * 50)
print(f"THÔNG TIN THỜI GIAN")
print(f"=" * 50)
print(f"Thời gian bắt đầu (UTC): {start_utc}")
print(f"Thời gian kết thúc (UTC): {end_utc}")
print(f"Thời lượng: {duration:.3f} giây")
print(f"Tần số lấy mẫu: {fs:.1f} Hz")
print(f"Khoảng cách mẫu: {np.mean(dt)*1000:.4f} ± {np.std(dt)*1000:.4f} ms")

In [ ]:
# Xem dữ liệu với thời gian UTC
df[['timestamp', 'datetime_utc', 'time_s', 'accX(g)', 'accY(g)', 'accZ(g)']].head(10)

## 3. Thống kê dữ liệu thô

In [ ]:
acc_cols = ['accX(g)', 'accY(g)', 'accZ(g)']

print("THỐNG KÊ DỮ LIỆU THÔ")
print("=" * 50)
for col in acc_cols:
    print(f"\n{col}:")
    print(f"  Min:  {df[col].min():.6f} g")
    print(f"  Max:  {df[col].max():.6f} g")
    print(f"  Mean: {df[col].mean():.6f} g")
    print(f"  Std:  {df[col].std():.6f} g")
    print(f"  RMS:  {np.sqrt(np.mean(df[col]**2)):.6f} g")

## 4. Trực quan hóa dữ liệu thô

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
labels = ['X', 'Y', 'Z']

for i, (col, color, label) in enumerate(zip(acc_cols, colors, labels)):
    axes[i].plot(df['time_s'], df[col], color=color, linewidth=0.5, alpha=0.8)
    axes[i].set_ylabel(f'Acc {label} (g)')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_title(f'Trục {label}: Mean={df[col].mean():.4f}g, RMS={np.sqrt(np.mean(df[col]**2)):.4f}g')

axes[-1].set_xlabel('Thời gian (s)')
fig.suptitle(f'Dữ liệu ADXL355 - Bắt đầu: {start_utc}', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Loại bỏ DC Offset

In [ ]:
print("LOẠI BỎ DC OFFSET")
print("=" * 50)

for col in acc_cols:
    offset = df[col].mean()
    df[col] = df[col] - offset
    print(f"{col}: offset = {offset:.6f} g")

## 6. Áp dụng bộ lọc

In [ ]:
from scipy import signal

def butterworth_filter(data, cutoff, fs, order=4, btype='low'):
    """Áp dụng Butterworth filter."""
    nyq = 0.5 * fs
    normalized_cutoff = cutoff / nyq
    if normalized_cutoff >= 1:
        return data
    b, a = signal.butter(order, normalized_cutoff, btype=btype)
    return signal.filtfilt(b, a, data)

# Cài đặt bộ lọc
highpass_cutoff = 0.5   # Hz - loại bỏ drift
lowpass_cutoff = 500    # Hz - loại bỏ nhiễu cao tần

print(f"Áp dụng bộ lọc:")
print(f"  High-pass: {highpass_cutoff} Hz")
print(f"  Low-pass: {lowpass_cutoff} Hz")

for col in acc_cols:
    # High-pass filter
    filtered = butterworth_filter(df[col].values, highpass_cutoff, fs, order=2, btype='high')
    # Low-pass filter
    filtered = butterworth_filter(filtered, lowpass_cutoff, fs, order=4, btype='low')
    df[f'{col}_filtered'] = filtered

print("Đã áp dụng bộ lọc!")

## 7. Tính Vector Magnitude

In [ ]:
# Vector magnitude từ dữ liệu gốc (đã loại bỏ offset)
df['acc_magnitude'] = np.sqrt(df['accX(g)']**2 + df['accY(g)']**2 + df['accZ(g)']**2)

# Vector magnitude từ dữ liệu đã lọc
df['acc_magnitude_filtered'] = np.sqrt(
    df['accX(g)_filtered']**2 + 
    df['accY(g)_filtered']**2 + 
    df['accZ(g)_filtered']**2
)

print(f"Vector Magnitude:")
print(f"  Mean: {df['acc_magnitude'].mean():.6f} g")
print(f"  Max:  {df['acc_magnitude'].max():.6f} g")
print(f"  RMS:  {np.sqrt(np.mean(df['acc_magnitude']**2)):.6f} g")

## 8. So sánh trước và sau khi lọc

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for i, (col, label) in enumerate(zip(acc_cols, labels)):
    axes[i].plot(df['time_s'], df[col], 'b-', linewidth=0.5, alpha=0.5, label='Gốc')
    axes[i].plot(df['time_s'], df[f'{col}_filtered'], 'r-', linewidth=0.5, alpha=0.8, label='Đã lọc')
    axes[i].set_ylabel(f'Acc {label} (g)')
    axes[i].legend(loc='upper right')
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Thời gian (s)')
fig.suptitle('So sánh dữ liệu trước và sau khi lọc', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Lưu dữ liệu đã xử lý

In [ ]:
# Sắp xếp lại các cột
cols_order = [
    'timestamp', 'datetime_utc', 'time_s',
    'accX(g)', 'accY(g)', 'accZ(g)', 'acc_magnitude',
    'accX(g)_filtered', 'accY(g)_filtered', 'accZ(g)_filtered', 'acc_magnitude_filtered'
]

df_output = df[cols_order]

# Lưu file
output_file = '../data/processed/adxl355_1_processed.csv'
df_output.to_csv(output_file, index=False)

print(f"Đã lưu file: {output_file}")
print(f"Số cột: {len(df_output.columns)}")
print(f"Số dòng: {len(df_output)}")

In [ ]:
# Xem trước dữ liệu đã xử lý
df_output.head(10)

## 10. Tóm tắt

Dữ liệu đã được xử lý với các bước:

1. **Chuyển đổi thời gian**: Unix timestamp → UTC datetime
2. **Thời gian tương đối**: Tính từ điểm bắt đầu (giây)
3. **Loại bỏ DC offset**: Trừ giá trị trung bình
4. **Lọc tín hiệu**:
   - High-pass: 0.5 Hz (loại bỏ drift)
   - Low-pass: 500 Hz (loại bỏ nhiễu cao tần)
5. **Vector magnitude**: Tính từ 3 trục

File đầu ra chứa:
- `timestamp`: Unix timestamp gốc
- `datetime_utc`: Thời gian UTC
- `time_s`: Thời gian tương đối (giây)
- `accX/Y/Z(g)`: Gia tốc đã loại bỏ offset
- `accX/Y/Z(g)_filtered`: Gia tốc đã lọc
- `acc_magnitude`: Vector magnitude